In [43]:
import yfinance as yf

In [44]:
RRR = 3

In [45]:
Ticker = "^GDAXI"

df = yf.download(Ticker, period="max", interval="1h")

df.columns.names = [None, None]

df.columns = df.columns.get_level_values(0)

df = df.drop(columns=["Volume"])

[*********************100%***********************]  1 of 1 completed


In [46]:
df["HofD"] = df.index.hour
df["Outcome"] = 0.0
df["Side"] = "None"

# df = df.tail(100) # for testing purposes

df.head()

,Close,High,Low,Open,HofD,Outcome,Side
Datetime,,,,,,,
2024-08-07 15:00:00+02:00,17650.310547,17655.029297,17566.890625,17574.189453,15,0.0,None
2024-08-07 16:00:00+02:00,17647.199219,17666.820312,17602.669922,17650.419922,16,0.0,None
2024-08-07 17:00:00+02:00,17610.779297,17648.769531,17604.160156,17647.199219,17,0.0,None
2024-08-08 09:00:00+02:00,17507.119141,17516.050781,17439.869141,17516.050781,9,0.0,None
2024-08-08 10:00:00+02:00,17541.619141,17551.599609,17467.710938,17507.179688,10,0.0,None


### Strategy

In [47]:
for index, root in df.iterrows():
    # candel nature (bear or bull)
    df.at[index, "Side"] = "Up" if root["Close"] > root["Open"] else "Down"

    risk = root["High"] - root["Low"]

    is_long = False
    is_short = False

    next_candles = df.loc[index:].iloc[1:]

    for _, nc in next_candles.iterrows():
        if not is_long and not is_short: # no trade has been triggered yet
            if nc["High"] >= root["High"] and nc["Low"] > root["Low"]: # Break to the upside
                # long trade triggered
                is_long = True

            elif nc["Low"] <= root["Low"] and nc["High"] < root["High"]: # Break to the downside
                # short trade triggered
                is_short = True

            else: break

        if is_long:
            if nc["Low"] <= root["Low"]: break

            if nc["High"] - root["High"] > RRR * risk:
                # take profit hit
                df.at[index, "Outcome"] = RRR
                break

        if is_short:
            if nc["High"] >= root["High"]: break

            if nc["Low"] - root["Low"] > RRR * risk:
                # take profit hit
                df.at[index, "Outcome"] = RRR
                break

In [48]:
result_df = (
    df.groupby("HofD")["Outcome"]
      .apply(lambda x: (x == RRR).mean())
      .rename("Win Rate")
      .reset_index()
).set_index("HofD")

result_df["Expectancy"] = (RRR + 1)* result_df["Win Rate"] - 1
result_df["Win Rate"] = result_df["Win Rate"] * 100

result_df

,Win Rate,Expectancy
HofD,,
9,13.861386,-0.445545
10,14.257426,-0.429703
11,14.624506,-0.415020
12,13.043478,-0.478261
13,13.043478,-0.478261
14,15.277778,-0.388889
15,19.603960,-0.215842
16,13.095238,-0.476190
17,34.325397,0.373016
